# Merge subcell loc

In [2]:
import polars as pl

In [23]:
gene_loc_df = pl.read_csv("hpa_subcellular_location_annotations/subcellular_location_data.tsv", separator="\t", infer_schema_length=10000000)
gene_loc_opencell_df = pl.read_csv("opencell_subcellular_location_annot/opencell-localization-annotations.csv", infer_schema_length=10000000)
gene_loc_all_df = gene_loc_df.select(
    pl.col(["Gene", "Gene name", "Main location", "Additional location", "Approved", "GO id"])
).join(
    gene_loc_opencell_df,
    left_on=["Gene name", "Gene"],
    right_on=["target_name", "ensg_id"],
    how="full",
    coalesce=True
).with_columns(
    pl.when(pl.col("Approved").is_not_null())
    .then(pl.col("Approved"))
    .when((pl.col("Approved").is_null())&(pl.col("annotations_grade_3").is_not_null()))
    .then(pl.col("annotations_grade_3"))
    .otherwise(pl.lit("Uncertain"))
    .alias("Metadata_merged_approved_Loc")
).rename(
    {
        "Main location": "Metadata_hpa_main_loc",
        "Additional location": "Metadata_hpa_additional_loc",
        "GO id": "Metadata_GO_ID",
        "Gene": "Metadata_Gene_ENSGID"
    }
).drop(
    ["annotations_grade_3","annotations_grade_2","annotations_grade_1","Approved"]
)

In [25]:
gene_loc_all_df

Metadata_Gene_ENSGID,Gene name,Metadata_hpa_main_loc,Metadata_hpa_additional_loc,Metadata_GO_ID,Metadata_merged_approved_Loc
str,str,str,str,str,str
"""ENSG00000000003""","""TSPAN6""","""Cell Junctions;Cytosol""","""Nucleoli fibrillar center""","""Cell Junctions (GO:0030054);Cy…","""Cell Junctions;Cytosol;Nucleol…"
"""ENSG00000000457""","""SCYL3""","""Cytosol;Golgi apparatus""",null,"""Cytosol (GO:0005829);Golgi app…","""Uncertain"""
"""ENSG00000000460""","""C1orf112""","""Nucleoplasm""","""Nucleoli""","""Nucleoli (GO:0005730);Nucleopl…","""Uncertain"""
"""ENSG00000000938""","""FGR""","""Plasma membrane""","""Aggresome""","""Aggresome (GO:0016235);Plasma …","""Aggresome;Plasma membrane"""
"""ENSG00000000971""","""CFH""","""Vesicles""",null,"""Vesicles (GO:0043231)""","""Vesicles"""
…,…,…,…,…,…
"""ENSG00000136021""","""SCYL2""",null,null,null,"""vesicles"""
"""ENSG00000115368""","""WDR75""",null,null,null,"""nucleolus_gc"""
"""ENSG00000114520""","""SNX4""",null,null,null,"""vesicles"""


In [26]:
gene_loc_all_df.write_csv("subcell_loc_hpa_opencell.tsv", separator="\t")